In [45]:
#!/usr/bin/env python
"""
Tunnel analysis Workflow

This script implements a workflow where you supply a receptor PDB file and a ligand file.
It creates a working subdirectory (named after the receptor), moves the PDB there,
modifies the CAVER configuration on the fly (adjusting first_frame, last_frame, and time_sparsity),
runs CAVER on the copied PDB (using the workdir as the -pdb option), and then for each tunnel found,
starts a separate CaverDock run using pyCaverDock.

Dependencies:
  - pyCaverDock (which provides classes like CaverDock, Receptor, Ligand, load_tunnel, discretize_tunnel, etc.)
  - pandas, matplotlib, py3Dmol, MDAnalysis
  - openbabel (via pybel)
"""

import os
import shutil
import subprocess
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob
import MDAnalysis as mda
import numpy as np
from scipy.spatial import Voronoi, ConvexHull, KDTree, Delaunay
from utils import utils
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis.coordinates.PDB")

# Import pyCaverDock API
from pycaverdock import (
    CaverDock,
    Receptor,
    Ligand,
    load_tunnel,
    discretize_tunnel,
    Direction,
    TrajectoryType,
    EnergyProfilePlot,
    plot_results
)
from pycaverdock.experiment import convert_eprofile_analysis
from pycaverdock.utils import get_basename

def prepare_protein(input_file):
    """Convert a PDB receptor to PDBQT using OpenBabel."""
    if os.path.splitext(input_file)[1].lower() != ".pdbqt":
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
        conv = ob.OBConversion()
        conv.SetInFormat("pdb")
        conv.SetOutFormat("pdbqt")
        conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
        obmol = ob.OBMol()
        conv.ReadFile(obmol, input_file)
        obmol.CorrectForPH()  # optional
        obmol.AddHydrogens()
        conv.WriteFile(obmol, output_file)
        return output_file
    return input_file

def prepare_ligand(input_file):
    """Convert a ligand (SDF or MOL2) to PDBQT using OpenBabel."""
    ext = os.path.splitext(input_file)[1].lower()[1:]
    base = os.path.splitext(input_file)[0]
    output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()
    mol.make3D()
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

class Miner:

    # Standard van der Waals radii for common atoms (in Angstroms)
    VDW_RADII = {
        'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'P': 1.80,
        'F': 1.47, 'CL': 1.75, 'BR': 1.85, 'I': 1.98, 'NA': 2.27, 'MG': 1.73,
        'K': 2.75, 'CA': 2.31, 'ZN': 1.39, 'FE': 1.80, 'DEFAULT': 1.70
    }

    def __init__(self, universe, caver_path=None, start_tolerance = 6):
        """
        universe = MDAnalysis universe
        caver_path = paths to the CAVER executables.
        start_tolerance = Size of ball in which atoms will be used to define caver starting point when using
                        method atom start in run_caver
        If not provided, they are assumed to be in subdirectories "caver" and "caverdock".
        """
        self.universe = universe
        self.start_tolerance = start_tolerance

        if caver_path:
            self.caver_path = caver_path  
        else:
            try:
                self.caver_path = os.path.abspath(f"{__file__}/caver")
            except:
                raise ValueError("you need to specify the absolute path to caver")

    def internal_pocket_detection(self, protein_selection):
        """
        Simple internal pocket detection algorithm based on alpha spheres.
        This is a simplified version of fpocket's methodology.
        """
        pockets = []
        protein_positions = protein_selection.positions
        protein_radii = np.array([self.VDW_RADII.get(atom.element.upper(), self.VDW_RADII['DEFAULT']) 
                                 for atom in protein_selection])
        
        # Use KDTree for faster nearest neighbor searches
        protein_tree = KDTree(protein_positions)
        
        # Use the Voronoi diagram to find potential alpha sphere centers
        vor = Voronoi(protein_positions)
        
        # Vectorized operations for alpha sphere detection
        alpha_spheres = []
        batch_size = 1000  # Process vertices in batches for memory efficiency
        
        for i in range(0, len(vor.vertices), batch_size):
            batch_vertices = vor.vertices[i:i+batch_size]
            
            # Find distances from vertices to atoms using KDTree for speed
            distances, indices = protein_tree.query(batch_vertices, k=4)
            
            # Calculate minimum distances considering atom radii
            min_distances = distances - protein_radii[indices]
            min_dist_per_vertex = np.min(min_distances, axis=1)
            max_dist_per_vertex = np.max(distances, axis=1)
            
            # Find vertices that qualify as alpha spheres
            valid_mask = (min_dist_per_vertex > 0) & (min_dist_per_vertex < 3.0) & (max_dist_per_vertex < 10.0)
            valid_indices = np.where(valid_mask)[0]
            
            for j in valid_indices:
                alpha_spheres.append({
                    'center': batch_vertices[j],
                    'radius': min_dist_per_vertex[j],
                    'nearest_atoms': indices[j]
                })
        
        # Step 2: Cluster alpha spheres into pockets
        if not alpha_spheres:
            return pockets
        
        # Faster clustering using KDTree
        if len(alpha_spheres) > 0:
            centers = np.array([sphere['center'] for sphere in alpha_spheres])
            center_tree = KDTree(centers)
            
            # Find clusters using radius neighbor query
            clusters = []
            already_clustered = set()
            
            for i, sphere in enumerate(alpha_spheres):
                if i in already_clustered:
                    continue
                    
                # Find all neighbors within clustering distance
                neighbors = center_tree.query_ball_point(sphere['center'], 3.0)
                
                # Create a new cluster
                new_cluster = [sphere]
                already_clustered.add(i)
                
                # Add neighbors to cluster
                for neighbor_idx in neighbors:
                    if neighbor_idx != i and neighbor_idx not in already_clustered:
                        new_cluster.append(alpha_spheres[neighbor_idx])
                        already_clustered.add(neighbor_idx)
                
                clusters.append(new_cluster)
        
        # Step 3: Convert clusters to pockets
        for cluster in clusters:
            if len(cluster) >= 3:  # Minimum alpha spheres to define a pocket
                centers = [sphere['center'] for sphere in cluster]
                pockets.append({
                    'center': np.mean(centers, axis=0),
                    'alpha_spheres': centers,
                    'size': len(centers)
                })
        
        return pockets

    def set_start_point_from_coordinates(self, coordinates):
        """
        Set the tunnel starting point using specified coordinates.
        
        Parameters:
        -----------
        coordinates : array-like
            3D coordinates [x, y, z] of the starting point
        """
        self.start_point = np.array(coordinates)
        return self.start_point
    
    def set_start_point_from_selection(self, selection_string):
        """
        Set the tunnel starting point as the center of a MDAnalysis selection.
        
        Parameters:
        -----------
        selection_string : str
            MDAnalysis selection string (e.g., 'resid 100' or 'name CA and resid 50-60')
        """
        selection = self.universe.select_atoms(selection_string)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{selection_string}' did not match any atoms")
            
        self.start_point = selection.center_of_mass()
        return self.start_point
    
    def set_start_point_from_pocket(self, selection_string, start_method, max_pocket_distance=5.0):
        """
        Set the tunnel starting point as the center of a pocket containing a specified selection.
        Uses fpocket or similar methodology to identify pockets.
        
        Parameters:
        -----------
        ligand_selection : str
            MDAnalysis selection string for ligand or region of interest
        max_pocket_distance : float
            Maximum distance (Å) between selection and pocket center to consider it "containing" the selection
            
        Returns:
        --------
        Tuple of 
            self.start_point - Coordinates of the pocket center, np.array
            self.start_atoms - list of atoms around the starting point
        """
        # Identify the ligand/selection of interest
        self.universe.trajectory[0]
        selection = self.universe.select_atoms(selection_string)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{selection_string}' did not match any atoms")
        
        selection_center = selection.center_of_mass()
        protein_selection = self.universe.select_atoms("protein")

        # Implementation of a simplified pocket detection algorithm
        # This is inspired by fpocket but implemented directly to avoid external dependencies
        pockets = self.internal_pocket_detection(protein_selection)
        
        # Find the closest pocket to the selection center
        closest_pocket = None
        min_distance = float('inf')
        
        for pocket in pockets:
            distance = np.linalg.norm(pocket['center'] - selection_center)
            if distance < min_distance and distance <= max_pocket_distance:
                min_distance = distance
                closest_pocket = pocket
        
        if closest_pocket is None:
            raise ValueError(f"No pocket found containing the selection '{selection_string}'")
        
        self.start_point = closest_pocket['center']
        atom_selection = self.universe.select_atoms(f"{selection_string} and point {utils.numpy_to_blankspace_sep_str(self.start_point)} {self.start_tolerance}")
        self.start_atoms = atom_selection.ids #ids return with first atom in universe at 1 whereas indicies start at 0.

        self.start_method = start_method

        return self.start_point, self.start_atoms

    def run_caver_tempdir(self, selection = "protein", start_frame=1, last_frame=1, sparsity=1):
        """
        Creates a tempdir and writes the specified number of frames to it for analysis.
        Then runs caver
        Only one protein can be analyzed at a time with this function.
        #TODO finish writing
        """
        # Get the directory where the script was originally run
        original_cwd = os.getcwd()
        
        # Create a temporary directory within the original working directory
        temp_dir = tempfile.mkdtemp(dir=original_cwd)

        atomgroup = self.universe.select_atoms(selection)
        analyzed_frames = []

        #write selection to PDB file
        if len(self.universe.trajectory) > 1:
            for ts in self.universe.trajectory[start_frame:last_frame:sparsity]:
                atomgroup.write(f"{temp_dir}/frame_{ts.frame}.pdb")
                analyzed_frames.append(f"{temp_dir}/frame_{ts.frame}.pdb")

        #write selection to PDB file if there is no trajectory (due to strange MDAnalysis error)
        elif len(self.universe.trajectory) == 1:
            atomgroup.write(f"{temp_dir}/frame_0.pdb")

        #where are we now?
        current_directory = os.getcwd()

        #we cd to avoid the annoying printing of whole_clustering
        os.chdir(temp_dir)
        self.tempdir = temp_dir

        self.caver_wrapper(temp_dir, start_frame, last_frame, sparsity)

        #TODO read in data to dataframe and then process the df into averages, prevalences and so on.

        #go back to where we came from
        os.chdir(current_directory)
        
        #Clean up the temporary directory after execution
        #shutil.rmtree(temp_dir, ignore_errors=True)
        #self.tempdir = None
    
    def caver_wrapper(self, protein_dir, start_frame=1, last_frame=1, sparsity=1):
        """
        Run CAVER on a directory of pdb files containing the same (aligned) structure and then reports back the results.
        This method requieres that a starting point function has been run first to set self.start_point and self.start_atoms
        You need to cd into the dir where caver shall be run to avoid a annoying file called whole_clustering.csv getting printed somewhere random.

        Method - "coord" or "atoms" this specifies if we are setting a coordinate (suitable for static structures) or a atom selection as
        the starting point for caver. We only generate this data from the first frame so make sure it is representative (eg by using pdb_avg from whatcat.md)
        """
        #try:
        # Read the default config file (assumed to be in the same dir as caver.jar)
        default_config_path = f"{self.caver_path}/config_default.txt"
        with open(default_config_path, "r") as f:
            config_lines = f.readlines()

        #Caver cannot handle last frame not being set or being set to a smnegative number (as is numpy convention)
        #but we can specify a arbitrarilly large number and then it will analyze all availible frames.
        if last_frame == -1 or last_frame == None:
            last_frame = 100000

        # Modify parameters: first_frame, last_frame, time_sparsity and starting point
        starting_point_set = False
        new_config_lines = []
        for line in config_lines:
            stripped = line.strip()
            if stripped.startswith("time_sparsity"):
                new_config_lines.append(f"time_sparsity {sparsity}\n")
            elif stripped.startswith("first_frame"):
                new_config_lines.append(f"first_frame {start_frame}\n")
            elif stripped.startswith("last_frame"):
                new_config_lines.append(f"last_frame {last_frame}\n")
            elif stripped.startswith("starting_point"):
                if starting_point_set == True:
                    pass
                elif starting_point_set == False:
                    if self.start_method == "coord":
                        new_config_lines.append(f"starting_point_coordinates {utils.numpy_to_blankspace_sep_str(self.start_point)}\n")
                    elif self.start_method == "atoms":
                        for atom in self.start_atoms:
                            new_config_lines.append(f"starting_point_atom {atom}\n")
                    starting_point_set = True

            else:
                new_config_lines.append(line)

        # Save modified config file in temp_dir
        new_config_path = os.path.join(protein_dir, "config.txt")
        with open(new_config_path, "w") as f:
            f.writelines(new_config_lines)

        # Build the CAVER command
        cmd = [
            "java", "-Xmx1200m", 
            "-jar", f"{self.caver_path}/caver.jar",
            "-home", f"{self.caver_path}/",
            "-pdb", f"{protein_dir}/",
            "-conf", new_config_path,
            "-out", f"{protein_dir}/"
        ]
        print("Running CAVER:", " ".join(cmd))
        subprocess.check_call(cmd)

        # Assume tunnels are saved as "tunnels.pdb" in temp_dir
        tunnels_file = os.path.join(protein_dir, "tunnels.pdb")
        return tunnels_file, protein_dir
        #except:
        #    return ValueError("Error in Caver excecution")


    def visualize_tunnel(self, receptor_file, tunnel_file, output_html):
        view = py3Dmol.view(width=800, height=600)
        with open(receptor_file, "r") as f:
            pdb_data = f.read()
        view.addModel(pdb_data, "pdb")
        with open(tunnel_file, "r") as f:
            tunnel_data = f.read()
        view.addModel(tunnel_data, "pdb")
        view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
        view.setStyle({'model': 1}, {'stick': {'color': 'red', 'radius': 0.3}})
        view.zoomTo()
        html_str = view._make_html()
        with open(output_html, "w") as f:
            f.write(html_str)
        print(f"Interactive visualization HTML saved to {output_html}")

if __name__ == "__main__":
    receptor_pdb = "1be0_final.pdb"    # Supply your receptor PDB file
    trajectory_file = "1be0_trajectory.dcd"
    ligand_file = "EtOH.sdf"      # Supply your ligand file (SDF or MOL2)

    #for devops only
    os.chdir("/home/erikna/compchem/WhatCat/miner")
    print('PWD is : \n', os.popen('pwd').read())
    
    universe = utils.load_universe(receptor_pdb, trajectory_file)

    miner = Miner(universe, caver_path="/home/erikna/compchem/WhatCat/miner/caver")

    #set start point
    miner.set_start_point_from_pocket("resid 124 or resid 289 or resid 175", start_method="atoms")
    miner.run_caver_tempdir(start_frame=1, last_frame=-1, sparsity=1)


PWD is : 
 /home/erikna/compchem/WhatCat/miner



/home/erikna/miniconda3/envs/whatcat/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Running CAVER: java -Xmx1200m -jar /home/erikna/compchem/WhatCat/miner/caver/caver.jar -home /home/erikna/compchem/WhatCat/miner/caver/ -pdb /home/erikna/compchem/WhatCat/miner/tmpq5r72nz9/ -conf /home/erikna/compchem/WhatCat/miner/tmpq5r72nz9/config.txt -out /home/erikna/compchem/WhatCat/miner/tmpq5r72nz9/
Caver computation started.
Merging all PDB files into one multimodel PDB file.
Settings loaded from:
/home/erikna/compchem/WhatCat/miner/tmpq5r72nz9/config.txt

Basic parameters:
starting_point_atom 1969 1973 1974 1975 1976 1977 1978 2746 2747 2748 2749 2750 2752 2753 2754 2755 2756 2757 4510 4512 4513 4514 4516 4517 4518 4519 4520
probe_radius 0.9
shell_radius 3.0
shell_depth 4.0
frame_weighting_coefficient 1.0
frame_clustering_threshold 1.0

*** Processing frame_1.pdb ***
0 redundant tunnels removed.
0 tunnels stored.
*** Processing frame_2.pdb ***
0 redundant tunnels removed.
0 tunnels stored.
*** Processing frame_3.pdb ***
0 redundant tunnels removed.
0 tunnels stored.
*** Proce

In [46]:
#TODO work on data collection and packaging for single frames as well as whole trajectories
#TODO start work on mutation script. Maybe use ligandMPNN and check if the residues 
# we can mutate are hydrogen bonded to try to avoid destabilizing mutations


In [65]:
import re
import os
from utils import utils
import glob

import importlib
importlib.reload(utils)

def read_caver_out(caver_out_dir):
    """
    This reads in data from the caver run to a df.
    Then compiles the data into a raw data df and a summary df. 
    """

    #gather frame number
    max_frame, min_frame, len_frame = utils.get_frame_num(caver_out_dir)

    #read csv
    bottleneck_df = utils.read_caver_bottleneck_csv(f"{caver_out_dir}/analysis/bottlenecks.csv")
    characteristics_df = pd.read_csv(f"{caver_out_dir}/analysis/tunnel_characteristics.csv", usecols=[" Length", " Curvature"])

    #remove leading spaces from column names
    characteristics_df.columns = characteristics_df.columns.str.strip()

    #concat the two csv:s
    bottleneck_df = pd.concat([bottleneck_df, characteristics_df], axis = 1)

    #read in tunnel coordinates from cluster_timeless
    #we do this as the tunnels are time averaged and does not include the unconnected point cloud found in /clusters
    bottleneck_df = pd.concat([bottleneck_df, utils.read_tunnel_coords(caver_out_dir)], axis = 1)

    #TODO implement averaging and sdev when there is more than one snapshot with a tunnel as well as computing prevalence %

    return bottleneck_df

read_caver_out(miner.tempdir)

,Snapshot,Tunnel cluster,Throughput,Cost,Bottleneck X,Bottleneck Y,Bottleneck Z,Bottleneck R,Bottleneck residues,Length,Curvature,coordinates,tunnel_point_radii
0,frame_6.pdb,1,0.542684,0.611227,33.077633,43.092822,28.324759,0.927623,"[A:262, A:289, A:175, A:263, A:260, A:194, A:1...",9.693471,1.064727,"[[32.84, 42.219, 30.569], [32.971, 42.345, 29....","[1.6, 1.28, 1.07, 0.93, 1.09, 1.06, 1.16, 1.37..."
1,frame_23.pdb,2,0.316641,1.149986,33.804841,43.619415,37.790981,1.016885,"[A:226, A:152, A:128, A:164, A:229, A:150, A:1...",19.952871,1.312613,"[[31.46, 42.968, 31.745], [31.541, 43.393, 32....","[1.6, 1.57, 1.51, 1.58, 1.42, 1.26, 1.05, 1.11..."
